In [ ]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from datasmith import setup_environment

setup_environment()

/mnt/sdd1/atharvas/formulacode/datasmith


In [ ]:
commit_pth = Path("scratch/artifacts/pipeflush/merge_commits_filtered.parquet")
commit_df = pd.read_parquet(commit_pth)
commit_df.head()

,sha,date,message,total_additions,total_deletions,total_files_changed,files_changed,patch,has_asv,file_change_summary,...,pr_review_comments_url,pr_review_comment_url,pr_comments_url,pr_statuses_url,pr_head,pr_base,pr__links,pr_author_association,pr_auto_merge,pr_active_lock_reason
0,27fdbb71cd0d566bdeb12746db59c9d908c6b5d5,2024-11-24T22:10:43+09:00,Merge pull request #70 from Natsume-Neko/powmo...,4,4,2,atcoder/math.py\ndocs/math.rst,From 2690515de03114a6bd1eb111de1241ec63e83799 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,"{'label': 'Natsume-Neko:powmod-wrapper', 'ref'...","{'label': 'not522:master', 'ref': 'master', 'r...",{'comments': {'href': 'https://api.github.com/...,CONTRIBUTOR,None,None
1,a30b7e590271d7b77459946695ae8ce984e50f0a,2021-05-16T06:51:02+09:00,Merge pull request #55 from not522/asv\n\nBenc...,210,18,7,.github/workflows/benchmarks.yml\n.gitignore\n...,From 02611141933200c8ef3f3c54dd872c6d812ee72b ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,"{'label': 'not522:asv', 'ref': 'asv', 'repo': ...","{'label': 'not522:master', 'ref': 'master', 'r...",{'comments': {'href': 'https://api.github.com/...,OWNER,None,None
2,ec7a06930c49e1f884f8a536ad5ed6a42675ed74,2024-11-24T21:58:51+09:00,Merge pull request #71 from not522/readthedocs...,16,0,1,.readthedocs.yaml,From 190f9d7a6648f12c804939c9e254f89eb92300d4 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,"{'label': 'not522:readthedocs-config', 'ref': ...","{'label': 'not522:master', 'ref': 'master', 'r...",{'comments': {'href': 'https://api.github.com/...,OWNER,None,None
3,2b05cbd2a55ac0fa1d2ada51e9892d9cd490f0b4,2024-11-22T20:19:20+09:00,Merge pull request #69 from Natsume-Neko/unify...,1,1,1,atcoder/lazysegtree.py,From 4083ef711499f308ebb527a07c927b5485856b15 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,"{'label': 'Natsume-Neko:unify-lazysegtree', 'r...","{'label': 'not522:master', 'ref': 'master', 'r...",{'comments': {'href': 'https://api.github.com/...,CONTRIBUTOR,None,None
4,ca69a20dad7d855f7af057aa36e124baa4ee0d6c,2024-11-20T10:52:50+09:00,Merge pull request #66 from not522/update-ci\n...,48,8,4,.github/workflows/benchmarks.yml\n.github/work...,From 215d05321cd292bd08544b7c6ae9a4f1ffcf38b0 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,https://api.github.com/repos/not522/ac-library...,"{'label': 'not522:update-ci', 'ref': 'update-c...","{'label': 'not522:master', 'ref': 'master', 'r...",{'comments': {'href': 'https://api.github.com/...,OWNER,None,None


In [3]:
import tiktoken

commit_df = commit_df.dropna(subset=["patch"])
print(commit_df.shape)
gpt4_encoding = tiktoken.get_encoding("o200k_base")
n_patch_tokens = gpt4_encoding.encode_batch(commit_df["patch"].fillna("").tolist(), num_threads=84)

(65494, 48)


In [ ]:
# filter out comments that are unlikely to be performance related
from datasmith.execution.filter_commits import crude_perf_filter

print(commit_df.shape)
filtered_df = crude_perf_filter(commit_df, filter_repos=False)
print(filtered_df.shape)

(65494, 48)


In [ ]:
# Add information about whether the dependencies in the commit can be installed with uv.
from dask import compute, delayed

from datasmith.execution.resolution import analyze_commit

analysis_tasks = [
    delayed(analyze_commit)(sha, repo)
    for sha, repo in filtered_df[["sha", "repo_name"]].itertuples(index=False, name=None)
]
extended_analysis = pd.DataFrame(list(compute(*analysis_tasks)))
filtered_extended_df = pd.concat([filtered_df, extended_analysis.add_prefix("analysis_")], axis=1)
filtered_extended_df = filtered_extended_df.dropna(subset=["analysis_can_install", "pr_base"]).query(
    "analysis_can_install"
)
filtered_extended_df = filtered_extended_df[
    ~filtered_extended_df["analysis_resolution_strategy"].str.startswith("unresolved")
]
repo_info = filtered_extended_df["pr_base"].apply(lambda d: pd.Series({**d["repo"], **{"sha": d["sha"]}}))
filtered_extended_df = pd.concat([filtered_extended_df, repo_info.add_prefix("pr_base_")], axis=1)
print(filtered_extended_df.shape)

(6, 146)


In [9]:
# collect data needed for building docker contexts
from datasmith.docker.context import Task
from datasmith.utils import _get_github_metadata


def date_to_unix_timestamp(date_str: str) -> int:
    from datetime import datetime, timezone

    dt = datetime.fromisoformat(date_str.replace("Z", "+00:00"))
    return int(dt.replace(tzinfo=timezone.utc).timestamp())


def make_task(row) -> str:
    owner, repo = row["repo_name"].split("/")
    sha = row["pr_merge_commit_sha"]
    # sha = row["pr_base_sha"]
    commit_date = date_to_unix_timestamp(row["pr_merged_at"])
    return Task(owner=owner, repo=repo, sha=sha, commit_date=commit_date).with_tag("run").get_image_name()


def get_patch_from_diff_url(row: pd.Series) -> str | None:
    repo_name = row["repo_name"]
    pull_number = row["pr_number"]
    endpoint = f"/repos/{repo_name}/pulls/{pull_number}"
    diff_text = _get_github_metadata(endpoint=endpoint, params={"diff_api": "true"})
    if not diff_text or "diff" not in diff_text:
        return None
    return diff_text["diff"]


filtered_extended_df["container_name"] = filtered_extended_df.apply(make_task, axis=1)
# filtered_extended_df["patch"] = filtered_extended_df.apply(get_patch_from_diff_url, axis=1)
patch_tasks = [delayed(get_patch_from_diff_url)(row) for _, row in filtered_extended_df.iterrows()]
filtered_extended_df["patch"] = list(compute(*patch_tasks))
filtered_extended_df = filtered_extended_df.dropna(subset=["patch", "container_name"])
print(filtered_extended_df.shape)

23:06:16 WARNING  simple_useragent.core: Falling back to historic user agent.


(6, 147)


In [ ]:
# save the dataframe
out_pth = Path("scratch/artifacts/pipeflush/merge_commits_filtered_with_patch.parquet")

# drop extra columns that cause issues with parquet
filtered_extended_df = filtered_extended_df.drop(
    columns=[
        "analysis_excluded_missing_on_pypi",
        "analysis_excluded_exists_incompatible",
        "analysis_excluded_other",
    ]
)

filtered_extended_df.to_parquet(out_pth, index=False)
print(f"Saved to {out_pth}")